## Publish a Pointset object

This example shows how to convert pointset data in CSV format into an Evo geoscience object using the Evo Python SDK.

### Requirements

You must have a Seequent account with the Evo entitlement to use this notebook.

The following parameters must be provided:

- The client ID of your Evo application.
- The callback/redirect URL of your Evo application.

To obtain these app credentials, refer to the [Apps and tokens guide](https://developer.seequent.com/docs/guides/getting-started/apps-and-tokens) in the Seequent Developer Portal.

In [1]:
from evo.notebooks import ServiceManagerWidget

input_path = "sample-data"

# Evo app credentials
client_id = "<your-client-id>"  # Replace with your client ID
redirect_url = "<your-redirect-url>"  # Replace with your redirect URL

manager = await ServiceManagerWidget.with_auth_code(
    redirect_url=redirect_url,
    client_id=client_id,
).login()

/Users/david.knight/Developer/evo-python-sdk-fork/code-samples/.venv/lib/python3.13/site-packages/evo/notebooks/authorizer.py:108: UserWarning: The evo.notebooks.AuthorizationCodeAuthorizer is not secure, and should only ever be used in Jupyter notebooks in a private environment.
  warnings.warn(


ServiceManagerWidget(children=(VBox(children=(HBox(children=(Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR…

### Load the typed PointSet API

In [2]:
%load_ext evo.widgets

from evo.objects.typed import EpsgCode, PointSet, PointSetData

### Load the input data

The typed PointSet API can build coordinates, attributes, and categorical lookup tables directly from a dataframe.

In [3]:
import pandas as pd

# Define the input CSV file.
input_file = f"{input_path}/WP_assay.csv"

# Load the point data into a pandas dataframe.
input_df = pd.read_csv(input_file)

input_df.head()

,X,Y,Z,CU_pct,AU_gpt,DENSITY,Hole ID
0,445198.861763,494110.588391,3052.607679,0.79,1.75,NaN,WP001
1,445200.147289,494110.577172,3051.075590,0.83,1.73,NaN,WP001
2,445201.432816,494110.565953,3049.543501,0.84,6.00,NaN,WP001
3,445202.718342,494110.554735,3048.011412,0.83,2.56,2.32,WP001
4,445204.003868,494110.543516,3046.479323,0.97,1.53,2.98,WP001


### Define object metadata

The typed objects API creates the pointset directly from a dataframe.

Enter values for these parameters:
- `object_name`: The name of the object.
- `object_path`: The file path where the object will be found.
- `object_epsg_code`: (Optional) The EPSG code for the coordinate reference system.
- `object_tags`: (Optional) A dictionary of tags to assign to the object.

In [4]:
# Define the basic object metadata.
object_name = "Pointset_SDK_demo"
object_path = "Jupyter_Example"

# Define the coordinate reference system (CRS) for the object.
# Use EpsgCode(...) for EPSG-based CRS values, or set this to None for an unspecified CRS.
object_epsg_code = EpsgCode(32650)

# Optional tags to attach to the published object.
object_tags = {"Source": "Jupyter Notebook", "Evo SDK": "0.1.5"}
object_description = "Pointset created from CSV data using the typed PointSet API."

# Define the full object path in the workspace.
full_obj_path = f"{object_path}/{object_name}.json"

### Prepare the locations dataframe

In [5]:
# The typed PointSet API expects x, y, z columns for coordinates.
# Any remaining columns are published as point attributes.
locations_df = input_df.rename(columns={"X": "x", "Y": "y", "Z": "z"}).copy()

# Cast Hole ID to category so the SDK publishes it as a categorical attribute
# backed by a lookup table rather than as a plain string column.
locations_df["Hole ID"] = locations_df["Hole ID"].astype("category")

# Keep the intended attribute order explicit before publishing.
locations_df = locations_df[["x", "y", "z", "CU_pct", "AU_gpt", "DENSITY", "Hole ID"]]

locations_df.head()

,x,y,z,CU_pct,AU_gpt,DENSITY,Hole ID
0,445198.861763,494110.588391,3052.607679,0.79,1.75,NaN,WP001
1,445200.147289,494110.577172,3051.075590,0.83,1.73,NaN,WP001
2,445201.432816,494110.565953,3049.543501,0.84,6.00,NaN,WP001
3,445202.718342,494110.554735,3048.011412,0.83,2.56,2.32,WP001
4,445204.003868,494110.543516,3046.479323,0.97,1.53,2.98,WP001


### Create the pointset data

In [6]:
# Create the data object used by the typed PointSet API.
# The locations dataframe contains both the coordinates and the point attributes.
pointset_data = PointSetData(
    name=object_name,
    description=object_description,
    locations=locations_df,
    coordinate_reference_system=object_epsg_code,
    tags=object_tags,
)

### Create and publish the pointset

In [8]:
# Create and publish the pointset at the requested workspace path.
# The typed API handles the schema conversion, data upload, and object creation.
pointset = await PointSet.create(manager, pointset_data, path=full_obj_path)

pointset

### Open the object in the Evo Viewer

In [9]:
from IPython.display import HTML, display

from evo.widgets import get_viewer_url_for_object

viewer_url = get_viewer_url_for_object(pointset)

display(
    HTML(
        f'<a href="{viewer_url}" target="_blank" rel="noopener noreferrer">Open object in the Evo Viewer</a> '
        '(opens in a new tab)'
    )
)

Success! You now have a new geoscience object in Evo containing your pointset data.

## Summary

In this example, we've completed the following:
* Loaded point data from CSV into a pandas dataframe.
* Renamed the coordinate columns to `x`, `y`, and `z` for the typed PointSet API.
* Cast `Hole ID` to pandas `category` so the SDK publishes it as a categorical attribute with a lookup table.
* Kept the attribute order explicit in the dataframe before publishing.
* Created and published the pointset directly with `PointSet.create(...)`.